In [ ]:
import polars as pl
from plotnine import *
import matplotlib.pyplot as plt

In [ ]:
anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/cadd_vep_annotations_processed_final_selected.parquet")
anno.select(['id', 'region']).collect()

In [ ]:
# anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/cadd_vep_annotations_processed_final_selected.parquet")

# cols = anno.collect_schema().names()
# rename_map = {}
# for c in cols:
#     new = c.lower()
#     if new.startswith('cadd_') and (not new.startswith('cadd_raw') and not new.startswith('cadd_phred')):
#         new = new[len('cadd_'):]
#     rename_map[c] = new

# anno = anno.rename(rename_map)
# # anno.sink_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated.parquet")
# anno.head().collect()

## Debug

In [ ]:
# abexp_df = pl.read_parquet("/s/project/abexp_veff/all_variant_combinations/hg38/predict/abexp_v1.1")
abexp_df = pl.scan_parquet("/s/public_webshare/public/abexp/hg38/chrom=chr14")
abexp_df.head().collect()

In [ ]:
abexp_df = pl.scan_parquet("/s/public_webshare/public/abexp/hg38/chrom=chr14")

variant_data = {
    "chrom": "chr14",
    "start": 31130178,
    "end": 31130179,
    "ref": "C",
    "alt": "A",
}
variant_df = pl.DataFrame([variant_data])

aesub = abexp_df.join(variant_df.lazy(), on=["chrom", "start", "end", "ref", "alt"], how="semi").collect(engine='streaming')
aesub

In [ ]:
t = aesub[['chrom', 'end', 'ref', 'alt', 'gene', 'tissue', 'tissue_type', 'abexp_v1.1']].with_columns(
    tissue = 'abexp_' + pl.col('tissue').str.replace_all(r"\(", "_").str.replace_all(r"\)", "").str.replace_all(' - ', '_').str.replace_all(' ', '_').str.to_lowercase()
).pivot(
    values='abexp_v1.1',
    index=['chrom', 'end', 'ref', 'alt', 'gene'],
    columns='tissue'
)
t

In [ ]:
anno = pl.read_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/cadd_vep_annotations_processed_final_selected.parquet", columns=['chrom', 'pos', 'ref', 'alt', 'region']).rename({'pos': 'end', 'region': 'gene'})
anno

In [ ]:
abexp_df = pl.scan_parquet("/s/public_webshare/public/abexp/hg38/chrom=chr14").select(['chrom', 'end', 'ref', 'alt', 'gene', 'tissue', 'abexp_v1.1'])

aesub = abexp_df.join(anno.lazy(), on=["chrom", "end", "ref", "alt", "gene"], how="semi").collect(engine='streaming').with_columns(
    tissue = 'abexp_' + pl.col('tissue').str.replace_all(r"\(", "_").str.replace_all(r"\)", "").str.replace_all(' - ', '_').str.replace_all(' ', '_').str.to_lowercase(),
    id = pl.col('chrom') + ':' + pl.col('end').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')
).pivot(
    values='abexp_v1.1',
    index=['id', 'chrom', 'end', 'ref', 'alt', 'gene'],
    on='tissue'
).rename({'end': 'pos', 'gene': 'region'})

aesub

## Actual computing

In [ ]:
from tqdm import tqdm

anno = pl.read_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated.parquet", columns=['chrom', 'pos', 'ref', 'alt', 'region']).rename({'pos': 'end', 'region': 'gene'})
chroms = anno["chrom"].unique().to_list()

abexp_pre = "/s/public_webshare/public/abexp/hg38/chrom="
op_dir = "/s/project/deeprvat/wgs_preprocessing_rap/annotation/abexp"

# Extract scores chromosome wise
for chrom in tqdm(chroms):
    scores_lazy = pl.scan_parquet(abexp_pre + chrom).select(['chrom', 'end', 'ref', 'alt', 'gene', 'tissue', 'abexp_v1.1'])
    print(f"Processing chromosome {chrom}")

    scores_lazy_chrom = scores_lazy.join(
        anno.filter(pl.col("chrom") == chrom).lazy(),
        on=["chrom", "end", "ref", "alt", "gene"], 
        how="semi"
    ).collect(engine='streaming').with_columns(
        tissue = 'abexp_' + pl.col('tissue').str.replace_all(r"\(", "_").str.replace_all(r"\)", "").str.replace_all(' - ', '_').str.replace_all(' ', '_').str.to_lowercase(),
        id = pl.col('chrom') + ':' + pl.col('end').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')
    ).pivot(
        values='abexp_v1.1',
        index=['id', 'chrom', 'end', 'ref', 'alt', 'gene'],
        on='tissue'
    ).rename({'end': 'pos', 'gene': 'region'})

    scores_lazy_chrom.write_parquet(f"{op_dir}/abexp_annotations_{chrom}.parquet")

# Merge all chromosome files
chroms_list_dfs = []
for chrom in tqdm(chroms):
    df = pl.scan_parquet(f"{op_dir}/abexp_annotations_{chrom}.parquet")
    chroms_list_dfs.append(df)

all_scores_df = pl.concat(chroms_list_dfs)
all_scores_df.sink_parquet(f"{op_dir}/abexp1.1_annotations_allchroms.parquet")

In [ ]:
# Merge all chromosome files
chroms_list_dfs = []
for chrom in tqdm(chroms):
    df = pl.scan_parquet(f"{op_dir}/abexp_annotations_{chrom}.parquet")
    chroms_list_dfs.append(df)

all_scores_df = pl.concat(chroms_list_dfs)
all_scores_df.sink_parquet(f"{op_dir}/abexp1.1_annotations_allchroms.parquet")

## Get max per variant

In [ ]:
op_dir = "/s/project/deeprvat/wgs_preprocessing_rap/annotation/abexp"
ae_vars = pl.scan_parquet(f"{op_dir}/abexp1.1_annotations_allchroms.parquet")
ae_vars.head().collect()

In [ ]:
abexp_columns = set(ae_vars.collect_schema().names()) - set(['id', 'chrom', 'pos', 'ref', 'alt', 'region'])
abexp_columns

In [ ]:
ae_vars.with_columns(
    abexp_abs_max = pl.max_horizontal([pl.col(c).abs() for c in abexp_columns])
).sink_parquet(f"{op_dir}/abexp1.1_annotations_all_ukb_vars_genebass1e6.parquet")

## Merge with bigg annotations file

In [ ]:
anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated.parquet")
anno.select(['id', 'region']).collect()

In [ ]:
ae_vars = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/abexp/abexp1.1_annotations_all_ukb_vars_genebass1e6.parquet")
ae_vars.head().collect()

In [ ]:
anno_sorted = anno.sort(['id', 'region'])
ae_vars_sorted = ae_vars.select(['id', 'region', 'abexp_abs_max']).sort(['id', 'region'])

tmp = anno_sorted.join(
    ae_vars_sorted,
    on=['id', 'region'],
    how='left',
)

tmp.sink_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250923.parquet", engine='streaming')

In [ ]:
anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250923.parquet")
anno.select(['id', 'region']).collect()

## PromoterAI abs

In [ ]:
anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250923.parquet")
anno.head().collect()

In [ ]:
plt.hist(anno.select(['promoterai']).collect(), bins=50, log=True)

In [ ]:
anno.with_columns(
    promoterai_abs = pl.col('promoterai').abs()
).sink_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250924.parquet", engine='streaming')